# 04 — Train scVI on the HVG8000 branch (disk-backed / memory-safe)

This notebook keeps the scientific configuration of the original scVI step:

- 351,427 cells;
- 8,000 HVGs;
- raw counts from `.X` (`layer=None`);
- `scvi_batch` as the batch key;
- `core_dataset` and `condition` as categorical covariates;
- mitochondrial/ribosomal percentages as continuous covariates when present;
- 30-dimensional scVI latent representation;
- negative-binomial likelihood;
- early stopping with a 90/10 train/validation split;
- neighbors, UMAP, and Leiden resolutions 0.1–0.5 on `X_scVI`.

## Memory-safety corrections

The input HVG matrix is never loaded fully into RAM. Instead:

1. the H5AD is validated directly with HDF5;
2. when space permits, one sequential copy is staged on the internal NVMe (`E:`) to avoid repeated USB I/O;
3. the staged H5AD is opened in backed mode;
4. a virtual `counts` layer is created in the NVMe staging H5AD as an HDF5 hard-link to `/X` (no count data are duplicated);
5. `AnnCollection` + scvi-tools `CollectionAdapter` streams that sparse layer during training;
6. GPU batch size starts conservatively at 256 and automatically retries at 128/64 only for CUDA OOM;
7. after latent extraction, the backed count file is closed before neighbors/UMAP;
8. latent/UMAP H5ADs retain all 8,000 gene metadata but intentionally omit the expression matrix. The full raw-count source remains the HVG input file.

The downstream pipeline transfers the scVI embeddings/clusters back to the
full-gene object, so duplicating a multi-GB count matrix in every scVI output
is unnecessary.

## scvi-tools 1.3.x compatibility

This environment exposes `CollectionAdapter` but not the newer AnnBatch setup API. In this scvi-tools generation, the documented AnnCollection workflow uses `setup_anndata(..., layer=<sparse layer>)`. `CollectionAdapter` does not provide a top-level `.X`, so the staged file exposes the existing raw `.X` matrix through a zero-copy HDF5 hard-link named `layers['counts']`. The durable HVG file remains unchanged.

## v5 fix — writable AnnCollection `.obsm`

scvi-tools stores joint categorical covariates (for example `core_dataset` and `condition`) in `obsm['_scvi_extra_categorical_covs']`, and continuous covariates in `obsm['_scvi_extra_continuous_covs']`. With the AnnCollection default `join_obsm=None`, the parent collection exposes `.obsm = None`, so `setup_anndata()` cannot create those fields. v5 constructs the collection with `join_obsm='inner'`; when no pre-existing obsm keys are present, AnnData creates an empty writable AxisArrays mapping, allowing scvi-tools to register the covariates without loading the expression matrix.

## v6 display compatibility fix

`model.view_anndata_setup()` is informational only. Some combinations of scvi-tools and Rich do not auto-import `rich.pretty`, causing an `AttributeError` after setup has already succeeded. v6 imports `rich.pretty` explicitly and treats failure of this optional display as non-fatal. Training configuration and scientific behavior are unchanged.

## v7 historical-training audit

The archived executed Ubuntu notebook was re-checked. Its actual training cell used `max_epochs=800`, `early_stopping=True`, monitor `elbo_validation`, patience 50, minimum delta 1e-3, and validation every 5 epochs. Although an earlier configuration cell recorded `BATCH_SIZE=512`, the executed `model.train(...)` call did **not** pass `batch_size`; therefore scvi-tools used its training default of 128. The archived run stopped at epoch 423/800 with the message that `elbo_validation` had not improved in the last 50 records. v7 restores that effective batch size explicitly while retaining the Windows/backed-memory compatibility fixes.


In [1]:
# =========================
# 0) Imports, paths, and runtime configuration
# =========================

import gc
import inspect
import json
import os
import random
import shutil
import sys
import time
import warnings
from pathlib import Path

# Set thread limits before numerical libraries do substantial work.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import scvi

from anndata.experimental import AnnCollection
from scvi.dataloaders import CollectionAdapter

try:
    from anndata.io import read_elem
except ImportError:
    from anndata._io.specs import read_elem

warnings.filterwarnings("default")
sc.settings.n_jobs = 4


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

INTEGRATED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
LOGS_DIR = PROJECT_DIR / "logs"

for directory in (
    INTEGRATED_DIR,
    METADATA_DIR,
    FIGURES_DIR,
    LOGS_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


N_HVG = 8000
RUN_TAG = f"HVG{N_HVG}_lowmem_Xcounts"

INPUT_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad"
)

LATENT_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_HVG8000_scVI_latent_lowmem_Xcounts.h5ad"
)
UMAP_LEIDEN_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_HVG8000_scVI_UMAP_Leiden_lowmem_Xcounts.h5ad"
)
MODEL_DIR = (
    INTEGRATED_DIR
    / "scVI_model_GBM_core_4datasets_HVG8000_lowmem_Xcounts"
)

# Compatibility aliases expected by later notebooks.
LATENT_ALIAS_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_HVG8000_scVI_latent.h5ad"
)
UMAP_ALIAS_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_HVG8000_scVI_UMAP_Leiden.h5ad"
)

SCVI_INPUT_QC_TSV = METADATA_DIR / "scVI_HVG8000_input_qc_lowmem.tsv"
SCVI_TRAINING_CONFIG_TSV = METADATA_DIR / "scVI_HVG8000_training_config_lowmem.tsv"
SCVI_HISTORY_TSV = METADATA_DIR / "scVI_HVG8000_training_history_lowmem.tsv"
SCVI_LATENT_QC_TSV = METADATA_DIR / "scVI_HVG8000_latent_qc_lowmem.tsv"
SCVI_CLUSTER_SIZES_TSV = METADATA_DIR / "scVI_HVG8000_leiden_cluster_sizes_lowmem.tsv"
SCVI_CLUSTER_FRACTIONS_TSV = METADATA_DIR / "scVI_HVG8000_cluster_dataset_condition_fractions_lowmem.tsv"
SCVI_OUTPUT_MANIFEST_TSV = METADATA_DIR / "scVI_HVG8000_output_manifest_lowmem.tsv"

EXPECTED_INPUT_SHAPE = (351_427, 8_000)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
scvi.settings.seed = SEED

torch.set_float32_matmul_precision("high")

N_LATENT = 30
N_HIDDEN = 128
N_LAYERS = 2
DROPOUT_RATE = 0.1
GENE_LIKELIHOOD = "nb"

MAX_EPOCHS = 800
EARLY_STOPPING_PATIENCE = 50
EARLY_STOPPING_MIN_DELTA = 1e-3
CHECK_VAL_EVERY_N_EPOCH = 5

# Conservative for RTX 4050 6 GB.
# Archived executed notebook omitted batch_size in model.train(), so the
# scvi-tools default (128) was the effective historical batch size.
# Use 64 only as a CUDA-OOM emergency fallback.
BATCH_SIZE_CANDIDATES = [128, 64]

ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES = 1

SCRATCH_ROOT = Path(
    os.environ.get(
        "GLIOMA_SCVI_SCRATCH_ROOT",
        r"E:\glioma-cnv-scratch\main_analysis_04_scvi",
    )
).expanduser().resolve()

SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

STAGED_INPUT = SCRATCH_ROOT / INPUT_H5AD.name
STAGED_INPUT_MARKER = SCRATCH_ROOT / f"{INPUT_H5AD.name}.source.json"

STAGED_LAYER_KEY = "counts"
STAGING_FORMAT_VERSION = "scvi-collection-hardlink-v4"

print("scvi-tools:", scvi.__version__)
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total GiB:", round(properties.total_memory / 1024**3, 2))

print("Input:", INPUT_H5AD)
print("Scratch:", SCRATCH_ROOT)
print("Accelerator:", ACCELERATOR)


C:\Users\AmirSilco\.conda\envs\glioma-cnv-atlas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 42


scvi-tools: 1.4.0.post1
scanpy: 1.11.5
anndata: 0.12.7
torch: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM total GiB: 6.0
Input: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad
Scratch: E:\glioma-cnv-scratch\main_analysis_04_scvi
Accelerator: gpu


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_14588\1429916693.py:164: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)
C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_14588\1429916693.py:165: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)


In [2]:
# =========================
# 1) File, HDF5, validation, and atomic-write helpers
# =========================

def remove_path(
    path: Path,
    attempts: int = 12,
    initial_delay_seconds: float = 0.25,
):
    path = Path(path)
    delay = float(initial_delay_seconds)
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            if path.is_dir():
                shutil.rmtree(path)
            elif path.exists():
                path.unlink()
            return

        except PermissionError as error:
            last_error = error
            gc.collect()

            if attempt == attempts:
                break

            print(
                "[Windows path-lock retry]",
                f"{attempt}/{attempts}",
                path.name,
                f"sleep={delay:.2f}s",
            )
            time.sleep(delay)
            delay = min(delay * 1.5, 2.0)

    raise PermissionError(
        f"Windows did not release the path lock: {path}"
    ) from last_error


def write_tsv_atomic(
    frame: pd.DataFrame,
    path: Path,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f"{path.name}.temporary")
    remove_path(temporary)

    frame.to_csv(
        temporary,
        sep="\t",
        index=False,
    )
    os.replace(temporary, path)
    print("[SAVED]", path)


def write_h5ad_atomic(
    adata_obj: ad.AnnData,
    path: Path,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f"{path.name}.temporary")
    remove_path(temporary)

    adata_obj.write_h5ad(
        temporary,
        compression="lzf",
    )

    remove_path(path)
    os.replace(temporary, path)
    print("[SAVED]", path)


def decode_h5_attribute(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return value


def stored_matrix_shape(node):
    if isinstance(node, h5py.Dataset):
        return tuple(int(value) for value in node.shape)

    shape = node.attrs.get("shape")
    if shape is None:
        raise KeyError(
            f"Sparse matrix group {node.name!r} has no shape."
        )

    return tuple(int(value) for value in shape)


def stored_sparse_encoding(group):
    return str(
        decode_h5_attribute(
            group.attrs.get("encoding-type", "")
        )
    )


def file_fingerprint(path: Path):
    stat = Path(path).stat()
    return {
        "size_bytes": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def virtual_counts_layer_is_valid(
    path: Path,
) -> bool:
    path = Path(path)

    if not path.exists():
        return False

    try:
        with h5py.File(path, "r") as handle:
            if "X" not in handle:
                return False

            if "layers" not in handle:
                return False

            if STAGED_LAYER_KEY not in handle["layers"]:
                return False

            x_node = handle["X"]
            layer_node = handle["layers"][STAGED_LAYER_KEY]

            if stored_matrix_shape(x_node) != stored_matrix_shape(layer_node):
                return False

            if not isinstance(x_node, h5py.Group):
                return False

            if not isinstance(layer_node, h5py.Group):
                return False

            if stored_sparse_encoding(x_node) != "csr_matrix":
                return False

            if stored_sparse_encoding(layer_node) != "csr_matrix":
                return False

            # Hard links resolve to the exact same HDF5 object.
            if x_node.id != layer_node.id:
                return False

        return True

    except Exception:
        return False


def ensure_virtual_counts_layer(
    path: Path,
):
    """
    Add layers['counts'] as an HDF5 hard-link to /X.

    This adds only HDF5 metadata; the count arrays are not copied.
    """
    path = Path(path)

    with h5py.File(path, "r+") as handle:
        if "X" not in handle:
            raise KeyError(
                f"{path}: /X is missing."
            )

        if not isinstance(handle["X"], h5py.Group):
            raise RuntimeError(
                f"{path}: /X must be stored as sparse CSR."
            )

        if stored_sparse_encoding(handle["X"]) != "csr_matrix":
            raise RuntimeError(
                f"{path}: /X is not csr_matrix."
            )

        if "layers" not in handle:
            layers_group = handle.create_group("layers")
            layers_group.attrs["encoding-type"] = "dict"
            layers_group.attrs["encoding-version"] = "0.1.0"
        else:
            layers_group = handle["layers"]

            if "encoding-type" not in layers_group.attrs:
                layers_group.attrs["encoding-type"] = "dict"

            if "encoding-version" not in layers_group.attrs:
                layers_group.attrs["encoding-version"] = "0.1.0"

        if STAGED_LAYER_KEY in layers_group:
            existing = layers_group[STAGED_LAYER_KEY]

            if existing.id != handle["X"].id:
                del layers_group[STAGED_LAYER_KEY]
                layers_group[STAGED_LAYER_KEY] = handle["X"]

        else:
            layers_group[STAGED_LAYER_KEY] = handle["X"]

        handle.flush()

    if not virtual_counts_layer_is_valid(path):
        raise AssertionError(
            "Failed to create a valid zero-copy counts hard-link."
        )


def staged_input_is_current():
    if not STAGED_INPUT.exists() or not STAGED_INPUT_MARKER.exists():
        return False

    try:
        with STAGED_INPUT_MARKER.open("r", encoding="utf-8") as handle:
            marker = json.load(handle)

        return (
            marker.get("source") == str(INPUT_H5AD.resolve())
            and marker.get("fingerprint") == file_fingerprint(INPUT_H5AD)
            and marker.get("format_version") == STAGING_FORMAT_VERSION
            and marker.get("virtual_layer_key") == STAGED_LAYER_KEY
            and virtual_counts_layer_is_valid(STAGED_INPUT)
        )

    except Exception:
        return False


def stage_input_for_collection_adapter():
    input_size = INPUT_H5AD.stat().st_size
    free_space = shutil.disk_usage(SCRATCH_ROOT).free

    # A staging copy is required so the durable H5AD remains untouched.
    required = input_size + 5 * 1024**3

    if free_space < required:
        raise OSError(
            "Not enough NVMe scratch space to create the scVI staging copy. "
            f"Need about {required / 1024**3:.1f} GiB free; observed "
            f"{free_space / 1024**3:.1f} GiB at {SCRATCH_ROOT}."
        )

    if staged_input_is_current():
        print(
            "[REUSING staged NVMe input with virtual counts layer]",
            STAGED_INPUT,
        )
        return STAGED_INPUT

    remove_path(STAGED_INPUT)
    remove_path(STAGED_INPUT_MARKER)

    print(
        "[STAGING sequential copy to NVMe]",
        INPUT_H5AD,
        "->",
        STAGED_INPUT,
    )

    shutil.copyfile(
        INPUT_H5AD,
        STAGED_INPUT,
    )

    if STAGED_INPUT.stat().st_size < input_size:
        remove_path(STAGED_INPUT)
        raise AssertionError(
            "Staged H5AD is unexpectedly smaller than the source."
        )

    print(
        "[ADDING zero-copy HDF5 layer alias]",
        f"layers[{STAGED_LAYER_KEY!r}] -> X",
    )

    ensure_virtual_counts_layer(
        STAGED_INPUT
    )

    marker = {
        "source": str(INPUT_H5AD.resolve()),
        "fingerprint": file_fingerprint(INPUT_H5AD),
        "format_version": STAGING_FORMAT_VERSION,
        "virtual_layer_key": STAGED_LAYER_KEY,
        "virtual_layer_is_hdf5_hardlink_to_X": True,
    }

    with STAGED_INPUT_MARKER.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            marker,
            handle,
            indent=2,
        )

    return STAGED_INPUT


def sample_raw_count_integrity(
    x_group: h5py.Group,
    max_values=1_000_000,
):
    if stored_sparse_encoding(x_group) != "csr_matrix":
        raise AssertionError(
            "The HVG input X must be stored as CSR."
        )

    data = x_group["data"]
    n_values = int(data.shape[0])

    if n_values == 0:
        raise ValueError("Input X has no nonzero values.")

    ranges = []
    window = min(max_values // 3, n_values)

    ranges.append((0, window))

    if n_values > window:
        middle = max(0, n_values // 2 - window // 2)
        ranges.append((middle, min(n_values, middle + window)))

    if n_values > 2 * window:
        ranges.append((n_values - window, n_values))

    minimum = np.inf
    maximum = -np.inf
    integer_like = True

    for start, end in ranges:
        values = np.asarray(data[start:end])

        if not np.isfinite(values).all():
            raise ValueError("Input X contains non-finite count values.")

        minimum = min(minimum, float(values.min()))
        maximum = max(maximum, float(values.max()))

        if np.issubdtype(values.dtype, np.integer):
            continue

        if not np.equal(values, np.floor(values)).all():
            integer_like = False

    return {
        "sampled_min": minimum,
        "sampled_max": maximum,
        "sampled_integer_like": bool(integer_like),
        "nnz": n_values,
        "dtype": str(data.dtype),
    }


def history_to_dataframe(history):
    if history is None:
        return pd.DataFrame()

    if isinstance(history, pd.DataFrame):
        result = history.copy()
        result = result.reset_index().rename(columns={"index": "epoch"})
        return result

    if isinstance(history, dict):
        pieces = []

        for metric, value in history.items():
            if isinstance(value, pd.DataFrame):
                frame = value.copy()
                if frame.shape[1] == 1:
                    frame = frame.rename(
                        columns={frame.columns[0]: metric}
                    )
                else:
                    frame = frame.add_prefix(f"{metric}__")
                pieces.append(frame)
            elif isinstance(value, pd.Series):
                pieces.append(value.rename(metric).to_frame())

        if not pieces:
            return pd.DataFrame()

        result = pd.concat(pieces, axis=1)
        result = result.reset_index().rename(columns={"index": "epoch"})
        return result

    try:
        return pd.DataFrame(history)
    except Exception:
        return pd.DataFrame()


In [3]:
# =========================
# 2) Validate HVG input without loading the expression matrix
# =========================

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Input HVG H5AD not found: {INPUT_H5AD}"
    )

with h5py.File(INPUT_H5AD, "r") as handle:
    if "X" not in handle:
        raise KeyError("Input H5AD is missing /X.")

    if "obs" not in handle or "var" not in handle:
        raise KeyError("Input H5AD is missing /obs or /var.")

    input_shape = stored_matrix_shape(handle["X"])

    if input_shape != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            f"Expected input shape {EXPECTED_INPUT_SHAPE}, "
            f"observed {input_shape}."
        )

    if not isinstance(handle["X"], h5py.Group):
        raise AssertionError(
            "Input X is dense; disk-backed sparse training requires CSR."
        )

    x_encoding = stored_sparse_encoding(handle["X"])
    count_qc = sample_raw_count_integrity(handle["X"])

    obs_metadata = read_elem(handle["obs"])
    var_metadata = read_elem(handle["var"])

    has_counts_layer = (
        "layers" in handle
        and "counts" in handle["layers"]
    )

if x_encoding != "csr_matrix":
    raise AssertionError(
        f"Input X must be csr_matrix; observed {x_encoding!r}."
    )

if has_counts_layer:
    raise AssertionError(
        "This low-memory HVG input unexpectedly contains layers['counts']. "
        "The scVI branch must use raw counts in X with layer=None."
    )

required_obs_cols = [
    "scvi_batch",
    "core_dataset",
    "condition",
    "sample_id",
    "patient_id",
]

missing = [
    column
    for column in required_obs_cols
    if column not in obs_metadata.columns
]

if missing:
    raise KeyError(
        f"Missing required scVI metadata columns: {missing}"
    )

if not obs_metadata.index.is_unique:
    raise AssertionError("Input cell identifiers are not unique.")

if not var_metadata.index.is_unique:
    raise AssertionError("Input gene identifiers are not unique.")

if count_qc["sampled_min"] < 0:
    raise ValueError("Input X contains negative values.")

if not count_qc["sampled_integer_like"]:
    raise ValueError(
        "Input X does not look like raw integer counts."
    )

categorical_covariates = [
    column
    for column in ("core_dataset", "condition")
    if column in obs_metadata.columns
]

continuous_covariates = [
    column
    for column in ("pct_counts_mt", "pct_counts_ribo")
    if column in obs_metadata.columns
]

for column in continuous_covariates:
    numeric = pd.to_numeric(
        obs_metadata[column],
        errors="coerce",
    )

    if numeric.isna().any():
        raise ValueError(
            f"Continuous covariate {column!r} contains missing/non-numeric values."
        )

print("Validated input shape:", input_shape)
print("Validated X encoding:", x_encoding)
print("Validated X dtype:", count_qc["dtype"])
print("X nonzero entries:", f"{count_qc['nnz']:,}")
print("Sampled X min/max:", count_qc["sampled_min"], count_qc["sampled_max"])
print("Sampled X integer-like:", count_qc["sampled_integer_like"])
print("counts layer:", "absent by design")
print("scvi_batch levels:", obs_metadata["scvi_batch"].nunique())
print("core_dataset levels:", obs_metadata["core_dataset"].nunique())
print("condition levels:", obs_metadata["condition"].nunique())
print("samples:", obs_metadata["sample_id"].nunique())
print("patients:", obs_metadata["patient_id"].nunique())

input_qc = pd.DataFrame(
    [
        {
            "input_h5ad": str(INPUT_H5AD),
            "run_tag": RUN_TAG,
            "n_obs": input_shape[0],
            "n_vars": input_shape[1],
            "x_encoding": x_encoding,
            "x_dtype": count_qc["dtype"],
            "x_nnz": count_qc["nnz"],
            "has_counts_layer": False,
            "counts_source_for_scvi": "X",
            "layer_for_setup_anndata": "None",
            "sampled_x_min": count_qc["sampled_min"],
            "sampled_x_max": count_qc["sampled_max"],
            "sampled_x_integer_like": count_qc["sampled_integer_like"],
            "batch_key": "scvi_batch",
            "categorical_covariate_keys": ",".join(categorical_covariates),
            "continuous_covariate_keys": ",".join(continuous_covariates),
            "n_scvi_batches": obs_metadata["scvi_batch"].nunique(),
            "n_core_datasets": obs_metadata["core_dataset"].nunique(),
            "n_conditions": obs_metadata["condition"].nunique(),
            "n_samples": obs_metadata["sample_id"].nunique(),
            "n_patients": obs_metadata["patient_id"].nunique(),
        }
    ]
)

write_tsv_atomic(
    input_qc,
    SCVI_INPUT_QC_TSV,
)

display(input_qc)


Validated input shape: (351427, 8000)
Validated X encoding: csr_matrix
Validated X dtype: int32
X nonzero entries: 316,259,948
Sampled X min/max: 1.0 4421.0
Sampled X integer-like: True
counts layer: absent by design
scvi_batch levels: 83
core_dataset levels: 4
condition levels: 4
samples: 83
patients: 50
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\scVI_HVG8000_input_qc_lowmem.tsv


,input_h5ad,run_tag,n_obs,n_vars,x_encoding,x_dtype,x_nnz,has_counts_layer,counts_source_for_scvi,layer_for_setup_anndata,...,sampled_x_max,sampled_x_integer_like,batch_key,categorical_covariate_keys,continuous_covariate_keys,n_scvi_batches,n_core_datasets,n_conditions,n_samples,n_patients
0,G:\Repository\glioma-cnv-single-cell-atlas\dat...,HVG8000_lowmem_Xcounts,351427,8000,csr_matrix,int32,316259948,False,X,None,...,4421.0,True,scvi_batch,"core_dataset,condition","pct_counts_mt,pct_counts_ribo",83,4,4,83,50


In [4]:
# =========================
# 3) Stage input on NVMe and construct a disk-backed CollectionAdapter
# =========================

TRAINING_INPUT = stage_input_for_collection_adapter()

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# The staged H5AD contains /X and /layers/counts pointing to the same
# HDF5 sparse CSR object.
backed_adata = sc.read_h5ad(
    TRAINING_INPUT,
    backed="r",
)

if tuple(backed_adata.shape) != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Backed input shape changed: {backed_adata.shape}"
    )

if STAGED_LAYER_KEY not in backed_adata.layers:
    raise AssertionError(
        f"Staged input is missing layers[{STAGED_LAYER_KEY!r}]."
    )

for column in (
    "scvi_batch",
    "core_dataset",
    "condition",
    "sample_id",
    "patient_id",
    "dataset_id",
    "geo_id",
    "study_id",
):
    if column in backed_adata.obs.columns:
        backed_adata.obs[column] = (
            backed_adata.obs[column]
            .astype(str)
            .astype("category")
        )

for column in continuous_covariates:
    backed_adata.obs[column] = pd.to_numeric(
        backed_adata.obs[column],
        errors="raise",
    ).astype(np.float32)

collection = AnnCollection(
    [backed_adata],
    join_vars="inner",
    join_obs="inner",
    # REQUIRED for scvi-tools joint categorical/continuous covariates.
    # With the default join_obsm=None, collection.obsm is None and
    # setup_anndata cannot create _scvi_extra_*_covs.
    join_obsm="inner",
)

if collection.obsm is None:
    raise AssertionError(
        "AnnCollection.obsm is None. "
        "join_obsm='inner' did not create the writable obsm mapping."
    )

print(
    "AnnCollection writable obsm keys before scVI setup:",
    list(collection.obsm.keys()),
)

adata_stream = CollectionAdapter(
    collection
)

print("Disk-backed training adapter:")
print(adata_stream)
print("isbacked:", adata_stream.isbacked)
print("training source:", TRAINING_INPUT)
print(
    "CollectionAdapter count source:",
    f"layers[{STAGED_LAYER_KEY!r}]",
)
print(
    "Scientific count source:",
    "raw X counts via zero-copy HDF5 hard-link",
)


[REUSING staged NVMe input with virtual counts layer] E:\glioma-cnv-scratch\main_analysis_04_scvi\GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad
AnnCollection writable obsm keys before scVI setup: []
Disk-backed training adapter:
Adapter for:
AnnCollection object with n_obs × n_vars = 351427 × 8000
  constructed from 1 AnnData objects
    view of layers: 'counts'
    obs: 'sample_id', 'geo_id', 'core_dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'dataset_id', 'study_id', 'source_dataset', 'source_file', 'concat_key', 'patient_id', 'condition', 'batch_dataset', 'batch_sample', 'batch_patient', 'original_cell_barcode', 'diagnosis', 'disease_stage', 'tumor_status', 'platform', 'technology', 'study_batch', 'sample_batch', 'scvi_batch', 'analysis_group_major', 'analysis_group_detail', 'ready_for_integration', 'cell_id_original'
isbacked: True
training source: E:\glioma-c

In [5]:
# =========================
# 4) Configure scVI on the zero-copy sparse counts layer
# =========================

BATCH_KEY = "scvi_batch"
CATEGORICAL_COVARIATE_KEYS = categorical_covariates
CONTINUOUS_COVARIATE_KEYS = continuous_covariates

# CollectionAdapter does not expose a top-level .X in this scvi-tools
# version. The documented AnnCollection path expects count data in a
# sparse layer. Here layers["counts"] is a zero-copy HDF5 hard-link to X.
scvi.model.SCVI.setup_anndata(
    adata_stream,
    layer=STAGED_LAYER_KEY,
    batch_key=BATCH_KEY,
    categorical_covariate_keys=(
        CATEGORICAL_COVARIATE_KEYS
        if CATEGORICAL_COVARIATE_KEYS
        else None
    ),
    continuous_covariate_keys=(
        CONTINUOUS_COVARIATE_KEYS
        if CONTINUOUS_COVARIATE_KEYS
        else None
    ),
)

# Verify the exact fields that previously failed to register.
if CATEGORICAL_COVARIATE_KEYS:
    if "_scvi_extra_categorical_covs" not in collection.obsm:
        raise AssertionError(
            "scVI did not register joint categorical covariates in "
            "AnnCollection.obsm."
        )

if CONTINUOUS_COVARIATE_KEYS:
    if "_scvi_extra_continuous_covs" not in collection.obsm:
        raise AssertionError(
            "scVI did not register joint continuous covariates in "
            "AnnCollection.obsm."
        )

print(
    "AnnCollection obsm keys after scVI setup:",
    list(collection.obsm.keys()),
)


def build_fresh_model():
    scvi.settings.seed = SEED
    torch.manual_seed(SEED)

    return scvi.model.SCVI(
        adata_stream,
        n_hidden=N_HIDDEN,
        n_latent=N_LATENT,
        n_layers=N_LAYERS,
        dropout_rate=DROPOUT_RATE,
        gene_likelihood=GENE_LIKELIHOOD,
    )


model = build_fresh_model()

training_config = pd.DataFrame(
    [
        {
            "run_tag": RUN_TAG,
            "n_hvg": N_HVG,
            "input_h5ad": str(INPUT_H5AD),
            "training_input": str(TRAINING_INPUT),
            "data_loader": (
                "AnnCollection+CollectionAdapter_backed_sparse_layer"
            ),
            "setup_anndata_layer": STAGED_LAYER_KEY,
            "scientific_counts_source": (
                "raw X counts; staged layers/counts is HDF5 hard-link to X"
            ),
            "durable_input_modified": False,
            "batch_key": BATCH_KEY,
            "categorical_covariate_keys": ",".join(
                CATEGORICAL_COVARIATE_KEYS
            ),
            "continuous_covariate_keys": ",".join(
                CONTINUOUS_COVARIATE_KEYS
            ),
            "n_hidden": N_HIDDEN,
            "n_latent": N_LATENT,
            "n_layers": N_LAYERS,
            "dropout_rate": DROPOUT_RATE,
            "gene_likelihood": GENE_LIKELIHOOD,
            "max_epochs": MAX_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
            "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
            "effective_historical_batch_size": 128,
            "historical_batch_size_reason": (
                "archived model.train omitted batch_size; "
                "scvi-tools training default was 128"
            ),
            "batch_size_candidates": ",".join(
                map(str, BATCH_SIZE_CANDIDATES)
            ),
            "accelerator": ACCELERATOR,
            "devices": DEVICES,
            "seed": SEED,
            "scvi_tools_version": scvi.__version__,
        }
    ]
)

write_tsv_atomic(
    training_config,
    SCVI_TRAINING_CONFIG_TSV,
)

print(
    "[SCVI SETUP COMPLETE]",
    f"layer={STAGED_LAYER_KEY!r}",
)
print(model)

# scvi-tools calls rich.pretty through the top-level rich module.
# Some Rich installations do not import that submodule automatically,
# which causes a cosmetic AttributeError after setup has already succeeded.
# Import it explicitly and keep the setup display non-fatal.
try:
    import rich.pretty  # noqa: F401

    model.view_anndata_setup()

except Exception as error:
    print(
        "[NON-FATAL setup-display compatibility warning]",
        repr(error),
    )
    print(
        "scVI setup itself completed successfully; "
        "only the optional rich-formatted registry display was skipped."
    )

display(training_config)


Seed set to 42


AnnCollection obsm keys after scVI setup: ['_scvi_extra_categorical_covs', '_scvi_extra_continuous_covs']
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\scVI_HVG8000_training_config_lowmem.tsv
[SCVI SETUP COMPLETE] layer='counts'


SCVI model with the following parameters: 
n_hidden: 128, n_latent: 30, n_layers: 2, dropout_rate: 0.1, dispersion: gene, gene_likelihood: nb, 
latent_distribution: normal.
Training status: Not Trained
Model's adata is minified?: False

Anndata setup with scvi-tools version 1.4.0.post1.

Setup via `SCVI.setup_anndata` with arguments:

...

                          batch State Registry                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃     Source Location     ┃      Categories       ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['scvi_batch'] │ GSM5518596_rGBM-01-A  │          0          │
│                         │ GSM5518597_rGBM-01-B  │          1          │
│                         │ GSM5518598_rGBM-01-C  │          2          │
│                         │ GSM5518599_rGBM-01-D  │          3          │
│                         │ GSM5518600_ndGBM-01-A │          4          │
│                         │ GSM5518601_ndGBM-01-C │          5          │
│                         │ GSM5518602_ndGBM-01-D │          6          │
│                         │ GSM5518603_ndGBM-01-F │          7          │
│                         │ GSM5518604_ndGBM-11-A │          8          │
│                         │ GSM5518605_ndGBM-11-B │          9          │
│                         │ GSM5518606_ndGBM-11-C │         10          │
│                         │ GSM5518607_ndGBM-11-D │         11          │
│                         │ GSM5518608_ndGBM-02-1 │         12          │
│                         │ GSM5518609_ndGBM-02-2 │         13          │
│                         │ GSM5518610_ndGBM-02-4 │         14          │
│                         │ GSM5518611_ndGBM-02-5 │         15          │
│                         │ GSM5518612_rGBM-02-2  │         16          │
│                         │ GSM5518613_rGBM-02-3  │         17          │
│                         │ GSM5518614_rGBM-02-4  │         18          │
│                         │ GSM5518615_rGBM-02-5  │         19          │
│                         │ GSM5518616_rGBM-03-1  │         20          │
│                         │ GSM5518617_rGBM-03-2  │         21          │
│                         │ GSM5518618_rGBM-03-3  │         22          │
│                         │ GSM5518619_rGBM-04-1  │         23          │
│                         │ GSM5518620_rGBM-04-2  │         24          │
│                         │ GSM5518621_rGBM-04-3  │         25          │
│                         │ GSM5518622_rGBM-04-4  │         26          │
│                         │ GSM5518623_ndGBM-03-1 │         27          │
│                         │ GSM5518624_ndGBM-03-2 │         28          │
│                         │ GSM5518625_ndGBM-03-3 │         29          │
│                         │ GSM5518626_rGBM-05-1  │         30          │
│                         │ GSM5518627_rGBM-05-2  │         31          │
│                         │ GSM5518628_rGBM-05-3  │         32          │
│                         │  GSM5518629_ndGBM-10  │         33          │
│                         │  GSM5518630_LGG-04-1  │         34          │
│                         │  GSM5518631_LGG-04-2  │         35          │
│                         │  GSM5518632_LGG-04-3  │         36          │
│                         │  GSM5518633_ndGBM-04  │         37          │
│                         │  GSM5518634_ndGBM-05  │         38          │
│                         │  GSM5518635_ndGBM-06  │         39          │
│                         │  GSM5518636_ndGBM-07  │         40          │
│                         │  GSM5518637_ndGBM-08  │         41          │
│                         │   GSM5518638_LGG-03   │         42          │
│                         │  GSM5518639_ndGBM-09  │         43          │
│                         │   GSM8546833_BT356    │         44          │
│                         │   GSM8546834_BT363    │         45          │
│                         │   GSM8546835_BT364    │         46          │
│                         │   GSM8546836_BT368    │         47          │
│                         │   GSM8546837_BT369    │         48          │
│                         │   GSM8546838_BT373    │         49          │
│   

                     labels State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['_scvi_labels'] │     0      │          0          │
└───────────────────────────┴────────────┴─────────────────────┘

               extra_categorical_covs State Registry               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃  Categories   ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│  adata.obs['condition']   │      GBM      │          0          │
│                           │      LGG      │          1          │
│                           │     ndGBM     │          2          │
│                           │     rGBM      │          3          │
│                           │               │                     │
│ adata.obs['core_dataset'] │ DS1_GSE103224 │          0          │
│                           │ DS2_GSE141383 │          1          │
│                           │ DS3_GSE182109 │          2          │
│                           │ DS4_GSE278450 │          3          │
│                           │               │                     │
└───────────────────────────┴───────────────┴─────────────────────┘

  extra_continuous_covs State   
            Registry            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃       Source Location        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  adata.obs['pct_counts_mt']  │
│ adata.obs['pct_counts_ribo'] │
└──────────────────────────────┘

,run_tag,n_hvg,input_h5ad,training_input,data_loader,setup_anndata_layer,scientific_counts_source,durable_input_modified,batch_key,categorical_covariate_keys,...,early_stopping_patience,early_stopping_min_delta,check_val_every_n_epoch,effective_historical_batch_size,historical_batch_size_reason,batch_size_candidates,accelerator,devices,seed,scvi_tools_version
0,HVG8000_lowmem_Xcounts,8000,G:\Repository\glioma-cnv-single-cell-atlas\dat...,E:\glioma-cnv-scratch\main_analysis_04_scvi\GB...,AnnCollection+CollectionAdapter_backed_sparse_...,counts,raw X counts; staged layers/counts is HDF5 har...,False,scvi_batch,"core_dataset,condition",...,50,0.001,5,128,archived model.train omitted batch_size; scvi-...,"128,64",gpu,1,42,1.4.0.post1


In [6]:
# =========================
# 5) Train scVI with archived effective training semantics
# =========================

# Historical executed Ubuntu run:
#   max_epochs = 800
#   train_size = 0.9
#   validation_size = 0.1
#   effective batch_size = 128 (model.train default; not explicitly passed)
#   early_stopping = True
#   monitor = elbo_validation
#   patience = 50
#   min_delta = 1e-3
#   check_val_every_n_epoch = 5
#
# v7 passes batch_size=128 explicitly so this effective setting cannot drift
# across future scvi-tools versions. 64 is used only after a genuine CUDA OOM.

def is_cuda_oom(error: BaseException) -> bool:
    text = str(error).lower()

    return (
        isinstance(
            error,
            getattr(
                torch.cuda,
                "OutOfMemoryError",
                RuntimeError,
            ),
        )
        or "cuda out of memory" in text
        or "cuda error: out of memory" in text
    )


def train_current_model(
    model_obj,
    batch_size: int,
):
    full_kwargs = {
        "max_epochs": MAX_EPOCHS,
        "train_size": 0.9,
        "validation_size": 0.1,
        "batch_size": int(batch_size),
        "accelerator": ACCELERATOR,
        "devices": DEVICES,
        "early_stopping": True,
        "early_stopping_monitor": "elbo_validation",
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
        "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
        "enable_progress_bar": True,
    }

    try:
        model_obj.train(
            **full_kwargs
        )

    except TypeError as error:
        # Compatibility with scvi-tools versions that do not expose one of
        # the explicit early-stopping keyword arguments.
        print(
            "[scvi-tools compatibility retry]",
            repr(error),
        )

        compatible_kwargs = {
            "max_epochs": MAX_EPOCHS,
            "train_size": 0.9,
            "validation_size": 0.1,
            "batch_size": int(batch_size),
            "accelerator": ACCELERATOR,
            "devices": DEVICES,
            "early_stopping": True,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
            "enable_progress_bar": True,
        }

        model_obj.train(
            **compatible_kwargs
        )


print("Archived effective training configuration:")
print("  max_epochs:", MAX_EPOCHS)
print("  train_size:", 0.9)
print("  validation_size:", 0.1)
print("  effective historical batch_size:", BATCH_SIZE_CANDIDATES[0])
print("  early_stopping:", True)
print("  early_stopping_monitor:", "elbo_validation")
print("  early_stopping_patience:", EARLY_STOPPING_PATIENCE)
print("  early_stopping_min_delta:", EARLY_STOPPING_MIN_DELTA)
print("  check_val_every_n_epoch:", CHECK_VAL_EVERY_N_EPOCH)
print()

selected_batch_size = None
last_error = None

for attempt_index, candidate_batch_size in enumerate(
    BATCH_SIZE_CANDIDATES,
    start=1,
):
    print()
    print(
        f"[TRAIN attempt {attempt_index}] "
        f"batch_size={candidate_batch_size}"
    )

    try:
        train_current_model(
            model,
            candidate_batch_size,
        )

        selected_batch_size = int(
            candidate_batch_size
        )
        break

    except Exception as error:
        last_error = error

        if (
            ACCELERATOR == "gpu"
            and is_cuda_oom(error)
            and candidate_batch_size
            != BATCH_SIZE_CANDIDATES[-1]
        ):
            print(
                "[CUDA OOM] Retrying from a fresh model "
                "with a smaller batch size."
            )

            del model
            gc.collect()
            torch.cuda.empty_cache()

            model = build_fresh_model()
            continue

        raise


if selected_batch_size is None:
    raise RuntimeError(
        "scVI training did not complete."
    ) from last_error


print()
print(
    "[TRAINING COMPLETE] batch_size:",
    selected_batch_size,
)


history_df = history_to_dataframe(
    model.history
)

if len(history_df):
    write_tsv_atomic(
        history_df,
        SCVI_HISTORY_TSV,
    )
    display(history_df.tail(20))
else:
    print(
        "[WARNING] Training completed but the current "
        "scvi-tools version did not expose a tabular history."
    )


Archived effective training configuration:
  max_epochs: 800
  train_size: 0.9
  validation_size: 0.1
  effective historical batch_size: 128
  early_stopping: True
  early_stopping_monitor: elbo_validation
  early_stopping_patience: 50
  early_stopping_min_delta: 0.001
  check_val_every_n_epoch: 5


[TRAIN attempt 1] batch_size=128


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\AmirSilco\.conda\envs\glioma-cnv-atlas\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.
C:\Users\AmirSilco\.conda\envs\glioma-cnv-atlas\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve perform

Epoch 400/800:  50%|█████     | 400/800 [6:46:22<6:46:22, 60.96s/it, v_num=1, train_loss=2.63e+3] 
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 2621.507. Signaling Trainer to stop.

[TRAINING COMPLETE] batch_size: 128
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\scVI_HVG8000_training_history_lowmem.tsv


,epoch,kl_weight,validation_loss,elbo_validation,reconstruction_loss_validation,kl_local_validation,kl_global_validation,train_loss,elbo_train,reconstruction_loss_train,kl_local_train,kl_global_train
380,380,0.95,2621.031494,2622.193359,2598.974854,23.21843,0.0,2625.153076,2626.383789,2601.675537,24.708239,0.0
381,381,0.9525,2620.731201,2621.82251,2598.871826,22.950567,0.0,2625.244629,2626.425049,2601.739746,24.685244,0.0
382,382,0.955,2621.342041,2622.37793,2599.330078,23.047796,0.0,2625.316162,2626.426025,2601.801758,24.624147,0.0
383,383,0.9575,2620.95459,2621.953613,2598.495117,23.458565,0.0,2625.339844,2626.385498,2601.813477,24.572004,0.0
384,384,0.96,2621.128418,2622.056152,2598.869141,23.186972,0.0,2625.338623,2626.320312,2601.7771,24.543184,0.0
385,385,0.9625,2621.26709,2622.134033,2598.979492,23.154644,0.0,2625.400146,2626.31665,2601.846924,24.469622,0.0
386,386,0.965,2621.198242,2622.0,2599.047363,22.952591,0.0,2625.435303,2626.290283,2601.852295,24.437899,0.0
387,387,0.9675,2621.643799,2622.388672,2599.468994,22.919628,0.0,2625.57251,2626.367188,2601.953369,24.413818,0.0
388,388,0.97,2621.497559,2622.192139,2599.034912,23.157156,0.0,2625.515137,2626.245605,2601.882568,24.363001,0.0
389,389,0.9725,2621.279541,2621.910156,2598.990723,22.919331,0.0,2625.683105,2626.347168,2602.022949,24.324221,0.0


In [7]:
# =========================
# 6) Save model and extract latent representation by streaming from disk
# =========================

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

latent = model.get_latent_representation(
    batch_size=selected_batch_size,
)

latent = np.asarray(
    latent,
    dtype=np.float32,
)

if latent.shape != (
    EXPECTED_INPUT_SHAPE[0],
    N_LATENT,
):
    raise AssertionError(
        f"Expected latent shape "
        f"({EXPECTED_INPUT_SHAPE[0]}, {N_LATENT}), "
        f"observed {latent.shape}."
    )

if not np.isfinite(latent).all():
    raise ValueError(
        "X_scVI contains non-finite values."
    )

# Save model without embedding/copying the large AnnData.
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)

model.save(
    MODEL_DIR,
    save_anndata=False,
    overwrite=True,
)

print("[SAVED MODEL]", MODEL_DIR)


latent_qc = pd.DataFrame(
    [
        {
            "run_tag": RUN_TAG,
            "n_obs": latent.shape[0],
            "n_latent": latent.shape[1],
            "X_scVI_min": float(np.min(latent)),
            "X_scVI_max": float(np.max(latent)),
            "X_scVI_mean": float(np.mean(latent)),
            "X_scVI_sd": float(np.std(latent)),
            "n_nan": int(np.isnan(latent).sum()),
            "training_batch_size": selected_batch_size,
        }
    ]
)

write_tsv_atomic(
    latent_qc,
    SCVI_LATENT_QC_TSV,
)

display(latent_qc)


# Build a lightweight downstream object:
# - preserves all 351,427 obs rows;
# - preserves all 8,000 var metadata rows;
# - does NOT duplicate the multi-GB expression matrix.
adata = ad.AnnData(
    X=None,
    obs=obs_metadata.copy(),
    var=var_metadata.copy(),
    shape=EXPECTED_INPUT_SHAPE,
)

adata.obsm["X_scVI"] = latent

adata.uns["scVI_run_info"] = {
    "run_tag": RUN_TAG,
    "n_hvg": int(N_HVG),
    "input_h5ad": str(INPUT_H5AD),
    "counts_source_for_training": (
        "raw X counts exposed as staged layers/counts HDF5 hard-link"
    ),
    "expression_matrix_in_latent_output": (
        "omitted by design; source retained in input_h5ad"
    ),
    "model_dir": str(MODEL_DIR),
    "batch_key": BATCH_KEY,
    "categorical_covariate_keys": (
        CATEGORICAL_COVARIATE_KEYS
    ),
    "continuous_covariate_keys": (
        CONTINUOUS_COVARIATE_KEYS
    ),
    "n_latent": int(N_LATENT),
    "n_hidden": int(N_HIDDEN),
    "n_layers": int(N_LAYERS),
    "dropout_rate": float(DROPOUT_RATE),
    "gene_likelihood": GENE_LIKELIHOOD,
    "seed": int(SEED),
    "training_batch_size": int(
        selected_batch_size
    ),
    "data_loader": (
        "AnnCollection+CollectionAdapter backed sparse layer"
    ),
}


write_h5ad_atomic(
    adata,
    LATENT_H5AD,
)
write_h5ad_atomic(
    adata,
    LATENT_ALIAS_H5AD,
)

print("Saved lightweight latent object:")
print(adata)


# Release model/count-file resources before graph construction.
del model
del adata_stream
del collection

try:
    backed_adata.file.close()
except Exception:
    pass

del backed_adata

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Backed expression file closed before neighbors/UMAP.")


[SAVED MODEL] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\scVI_model_GBM_core_4datasets_HVG8000_lowmem_Xcounts
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\scVI_HVG8000_latent_qc_lowmem.tsv


,run_tag,n_obs,n_latent,X_scVI_min,X_scVI_max,X_scVI_mean,X_scVI_sd,n_nan,training_batch_size
0,HVG8000_lowmem_Xcounts,351427,30,-7.453981,6.459757,-0.000262,0.741598,0,128


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_scVI_latent_lowmem_Xcounts.h5ad
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_HVG8000_scVI_latent.h5ad
Saved lightweight latent object:
AnnData object with n_obs × n_vars = 351427 × 8000
    obs: 'sample_id', 'geo_id', 'core_dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'dataset_id', 'study_id', 'source_dataset', 'source_file', 'concat_key', 'patient_id', 'condition', 'batch_dataset', 'batch_sample', 'batch_patient', 'original_cell_barcode', 'diagnosis', 'disease_stage', 'tumor_status', 'platform', 'technology', 'study_batch', 'sample_batch', 'scvi_batch', 'analysis_group_major', 'analysis_group_detail', 'ready_for_integration', 'cell_id_original'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_no

In [8]:
# =========================
# 7) Neighbors, UMAP, and Leiden on X_scVI only
# =========================

if "X_scVI" not in adata.obsm:
    raise KeyError(
        "X_scVI is missing from the lightweight latent object."
    )

N_NEIGHBORS = 15
LEIDEN_RESOLUTIONS = [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
]

print("Computing neighbors on X_scVI...")

neighbors_kwargs = {
    "use_rep": "X_scVI",
    "n_neighbors": N_NEIGHBORS,
    "random_state": SEED,
    "key_added": "neighbors_scVI",
}

# Prefer explicit approximate nearest neighbors when supported.
if "transformer" in inspect.signature(
    sc.pp.neighbors
).parameters:
    neighbors_kwargs["transformer"] = "pynndescent"

sc.pp.neighbors(
    adata,
    **neighbors_kwargs,
)

print("Computing UMAP...")

sc.tl.umap(
    adata,
    neighbors_key="neighbors_scVI",
    random_state=SEED,
)

if (
    "X_umap" not in adata.obsm
    or adata.obsm["X_umap"].shape
    != (EXPECTED_INPUT_SHAPE[0], 2)
):
    raise AssertionError(
        "UMAP did not produce the expected "
        "(351427, 2) embedding."
    )

cluster_rows = []

for resolution in LEIDEN_RESOLUTIONS:
    key = f"leiden_scvi_{resolution}"

    print("Running", key)

    sc.tl.leiden(
        adata,
        resolution=resolution,
        key_added=key,
        flavor="igraph",
        n_iterations=2,
        directed=False,
        random_state=SEED,
        neighbors_key="neighbors_scVI",
    )

    sizes = (
        adata.obs[key]
        .astype(str)
        .value_counts()
        .rename_axis("cluster")
        .reset_index(name="n_cells")
    )

    sizes["resolution"] = key
    sizes["run_tag"] = RUN_TAG
    cluster_rows.append(sizes)

cluster_sizes = pd.concat(
    cluster_rows,
    ignore_index=True,
)[
    [
        "run_tag",
        "resolution",
        "cluster",
        "n_cells",
    ]
]

write_tsv_atomic(
    cluster_sizes,
    SCVI_CLUSTER_SIZES_TSV,
)

display(cluster_sizes.head(20))


Computing neighbors on X_scVI...
Computing UMAP...
Running leiden_scvi_0.1
Running leiden_scvi_0.2
Running leiden_scvi_0.3
Running leiden_scvi_0.4
Running leiden_scvi_0.5
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\scVI_HVG8000_leiden_cluster_sizes_lowmem.tsv


,run_tag,resolution,cluster,n_cells
0,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,0,169514
1,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,1,137877
2,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,5,21632
3,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,2,14936
4,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,3,6266
5,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,4,1202
6,HVG8000_lowmem_Xcounts,leiden_scvi_0.2,0,167541
7,HVG8000_lowmem_Xcounts,leiden_scvi_0.2,2,129155
8,HVG8000_lowmem_Xcounts,leiden_scvi_0.2,6,23892
9,HVG8000_lowmem_Xcounts,leiden_scvi_0.2,4,14974


In [9]:
# =========================
# 8) Cluster composition reports
# =========================

fraction_rows = []

for resolution in [
    f"leiden_scvi_{value}"
    for value in LEIDEN_RESOLUTIONS
]:
    for metadata_column in (
        "core_dataset",
        "condition",
        "sample_id",
        "patient_id",
    ):
        table = pd.crosstab(
            adata.obs[resolution].astype(str),
            adata.obs[metadata_column].astype(str),
            normalize="index",
        )

        long = (
            table
            .reset_index()
            .melt(
                id_vars=resolution,
                var_name="metadata_value",
                value_name="fraction",
            )
            .rename(
                columns={
                    resolution: "cluster",
                }
            )
        )

        long["resolution"] = resolution
        long["metadata_column"] = metadata_column
        long["run_tag"] = RUN_TAG

        fraction_rows.append(
            long[
                [
                    "run_tag",
                    "resolution",
                    "metadata_column",
                    "metadata_value",
                    "cluster",
                    "fraction",
                ]
            ]
        )


fractions = pd.concat(
    fraction_rows,
    ignore_index=True,
)

write_tsv_atomic(
    fractions,
    SCVI_CLUSTER_FRACTIONS_TSV,
)

display(fractions.head(20))


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\scVI_HVG8000_cluster_dataset_condition_fractions_lowmem.tsv


,run_tag,resolution,metadata_column,metadata_value,cluster,fraction
0,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS1_GSE103224,0,0.115300
1,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS1_GSE103224,1,0.017574
2,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS1_GSE103224,2,0.011449
3,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS1_GSE103224,3,0.154006
4,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS1_GSE103224,4,0.000000
5,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS1_GSE103224,5,0.004761
6,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS2_GSE141383,0,0.108298
7,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS2_GSE141383,1,0.013265
8,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS2_GSE141383,2,0.040439
9,HVG8000_lowmem_Xcounts,leiden_scvi_0.1,core_dataset,DS2_GSE141383,3,0.092563


In [10]:
# =========================
# 9) Save lightweight UMAP/Leiden outputs and manifest
# =========================

adata.uns["scVI_umap_leiden_run_info"] = {
    "neighbors_key": "neighbors_scVI",
    "n_neighbors": int(N_NEIGHBORS),
    "leiden_resolutions": LEIDEN_RESOLUTIONS,
    "umap_random_state": int(SEED),
    "expression_matrix": (
        "omitted by design; use source HVG H5AD for counts"
    ),
}

write_h5ad_atomic(
    adata,
    UMAP_LEIDEN_H5AD,
)
write_h5ad_atomic(
    adata,
    UMAP_ALIAS_H5AD,
)


# Read-only validation; AnnData is not used as a context manager.
check = sc.read_h5ad(
    UMAP_LEIDEN_H5AD,
    backed="r",
)

try:
    if tuple(check.shape) != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            f"Saved UMAP object has shape {check.shape}, "
            f"expected {EXPECTED_INPUT_SHAPE}."
        )

    if "X_scVI" not in check.obsm:
        raise AssertionError(
            "Saved UMAP object is missing obsm['X_scVI']."
        )

    if "X_umap" not in check.obsm:
        raise AssertionError(
            "Saved UMAP object is missing obsm['X_umap']."
        )

    for resolution in LEIDEN_RESOLUTIONS:
        column = f"leiden_scvi_{resolution}"

        if column not in check.obs:
            raise AssertionError(
                f"Saved UMAP object is missing obs[{column!r}]."
            )

finally:
    check.file.close()


manifest = pd.DataFrame(
    [
        {
            "name": "input_h5ad",
            "path": str(INPUT_H5AD),
            "exists": INPUT_H5AD.exists(),
        },
        {
            "name": "latent_h5ad",
            "path": str(LATENT_H5AD),
            "exists": LATENT_H5AD.exists(),
        },
        {
            "name": "latent_alias_h5ad",
            "path": str(LATENT_ALIAS_H5AD),
            "exists": LATENT_ALIAS_H5AD.exists(),
        },
        {
            "name": "umap_leiden_h5ad",
            "path": str(UMAP_LEIDEN_H5AD),
            "exists": UMAP_LEIDEN_H5AD.exists(),
        },
        {
            "name": "umap_leiden_alias_h5ad",
            "path": str(UMAP_ALIAS_H5AD),
            "exists": UMAP_ALIAS_H5AD.exists(),
        },
        {
            "name": "model_dir",
            "path": str(MODEL_DIR),
            "exists": MODEL_DIR.exists(),
        },
        {
            "name": "training_input_used",
            "path": str(TRAINING_INPUT),
            "exists": Path(TRAINING_INPUT).exists(),
        },
    ]
)

write_tsv_atomic(
    manifest,
    SCVI_OUTPUT_MANIFEST_TSV,
)

display(manifest)

print()
print("[DONE] scVI training and graph construction completed successfully.")
print("Validated cells:", adata.n_obs)
print("Validated model genes metadata:", adata.n_vars)
print("Validated X_scVI shape:", adata.obsm["X_scVI"].shape)
print("Validated X_umap shape:", adata.obsm["X_umap"].shape)
print(
    "Validated Leiden columns:",
    [
        column
        for column in adata.obs.columns
        if column.startswith("leiden_scvi_")
    ],
)
print("Training batch size used:", selected_batch_size)
print("Expression matrix in latent/UMAP outputs: omitted by design")


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_scVI_UMAP_Leiden_lowmem_Xcounts.h5ad
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_HVG8000_scVI_UMAP_Leiden.h5ad
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\scVI_HVG8000_output_manifest_lowmem.tsv


,name,path,exists
0,input_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
1,latent_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
2,latent_alias_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
3,umap_leiden_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
4,umap_leiden_alias_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
5,model_dir,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
6,training_input_used,E:\glioma-cnv-scratch\main_analysis_04_scvi\GB...,True



[DONE] scVI training and graph construction completed successfully.
Validated cells: 351427
Validated model genes metadata: 8000
Validated X_scVI shape: (351427, 30)
Validated X_umap shape: (351427, 2)
Validated Leiden columns: ['leiden_scvi_0.1', 'leiden_scvi_0.2', 'leiden_scvi_0.3', 'leiden_scvi_0.4', 'leiden_scvi_0.5']
Training batch size used: 128
Expression matrix in latent/UMAP outputs: omitted by design


In [11]:
# =========================
# 10) Optional lightweight UMAP plots and scratch cleanup
# =========================

PLOT_FIGURES = False

if PLOT_FIGURES:
    sc.settings.figdir = str(FIGURES_DIR)

    plot_columns = [
        "core_dataset",
        "condition",
        "leiden_scvi_0.2",
        "leiden_scvi_0.3",
        "leiden_scvi_0.4",
    ]

    sc.pl.umap(
        adata,
        color=[
            column
            for column in plot_columns
            if column in adata.obs.columns
        ],
        ncols=2,
        wspace=0.4,
        frameon=False,
        show=False,
        save=f"_{RUN_TAG}_core_dataset_condition_leiden.png",
    )

    print("[SAVED UMAP FIGURES]", FIGURES_DIR)

else:
    print("PLOT_FIGURES=False; skipping plots.")


# The staged H5AD contains the temporary zero-copy counts-layer alias.
# Remove it only after all durable outputs have been validated.
if TRAINING_INPUT == STAGED_INPUT:
    try:
        remove_path(STAGED_INPUT)
        remove_path(STAGED_INPUT_MARKER)
        print("[CLEANED NVMe staged input]", STAGED_INPUT)
    except PermissionError as error:
        print(
            "[WARNING] Could not remove staged input yet:",
            repr(error),
        )

gc.collect()

print()
print("04_train_scvi finished.")


PLOT_FIGURES=False; skipping plots.
[CLEANED NVMe staged input] E:\glioma-cnv-scratch\main_analysis_04_scvi\GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad

04_train_scvi finished.


## Expected successful completion

The key checkpoints are:

```text
[ADDING zero-copy HDF5 layer alias] layers['counts'] -> X
CollectionAdapter count source: layers['counts']
AnnCollection writable obsm keys before scVI setup: ...
AnnCollection obsm keys after scVI setup: ['_scvi_extra_categorical_covs', ...]
[SCVI SETUP COMPLETE] layer='counts'
[TRAINING COMPLETE] batch_size: ...
[DONE] scVI training and graph construction completed successfully.
Validated cells: 351427
Validated model genes metadata: 8000
Validated X_scVI shape: (351427, 30)
Validated X_umap shape: (351427, 2)
Validated Leiden columns: [...]
Expression matrix in latent/UMAP outputs: omitted by design
```

The durable HVG input is **not modified**. Raw counts remain in `.X`.

Only the temporary NVMe staging copy contains `layers["counts"]`, implemented
as an HDF5 hard-link to the same `/X` CSR object, so no second multi-GB count
matrix is created.

This compatibility path is intended for scvi-tools versions where
`CollectionAdapter` is available but the newer AnnBatch setup API is not.

### Historical early-stopping reference

The archived executed Ubuntu notebook stopped at approximately:

```text
Epoch 423/800
Monitored metric elbo_validation did not improve in the last 50 records.
Best score: 2625.125. Signaling Trainer to stop.
```

The exact stopping epoch can vary slightly with hardware/data-loader execution, but v7 restores the same effective training batch size (128) and the same early-stopping parameters.
